# ENSO Forecasting: Ensemble Regression Models

This notebook trains two ensemble regressors for next-month ENSO sea surface temperature prediction using the 12-month sliding-window features:

- Random Forest Regressor
- XGBoost Regressor

The original chronological train/test split is preserved, and both models are evaluated on training and test sets using RMSE, MAE, and R^2.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

## 2. Load the Data

In [ ]:
# This works whether the notebook is run from the notebooks folder or the project root.
data_dir = Path("../data")
if not data_dir.exists():
    data_dir = Path("data")

X_train = pd.read_csv(data_dir / "X_train.csv")
X_test = pd.read_csv(data_dir / "X_test.csv")
y_train = pd.read_csv(data_dir / "y_train.csv").to_numpy(dtype=np.float32).ravel()
y_test = pd.read_csv(data_dir / "y_test.csv").to_numpy(dtype=np.float32).ravel()

print("Loaded data from:", data_dir.resolve())
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 3. Prepare Data for Model

In [ ]:
total_samples = len(X_train) + len(X_test)
train_pct = len(X_train) / total_samples * 100
test_pct = len(X_test) / total_samples * 100

print(f"Total samples: {total_samples}")
print(f"Train samples: {len(X_train)} ({train_pct:.1f}%)")
print(f"Test samples: {len(X_test)} ({test_pct:.1f}%)")
print(f"Features: {list(X_train.columns)}")
print("Target: target_t")
print("Split method: chronological, no shuffle")

## 4. Build the Models

In [ ]:
models = {
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "XGBoost Regressor": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        objective="reg:squarederror"
    )
}

models

## 5. Train the Models

In [ ]:
for model_name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {model_name}")

## 6. Evaluate on the Test Set

In [ ]:
def evaluate_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

metrics_rows = []
predictions_df = pd.DataFrame({"y_actual": y_test})

for model_name, model in models.items():
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_rmse, train_mae, train_r2 = evaluate_regression(y_train, y_pred_train)
    test_rmse, test_mae, test_r2 = evaluate_regression(y_test, y_pred_test)

    metrics_rows.append({
        "Model": model_name,
        "RMSE_Train": train_rmse,
        "MAE_Train": train_mae,
        "R2_Train": train_r2,
        "RMSE_Test": test_rmse,
        "MAE_Test": test_mae,
        "R2_Test": test_r2,
    })

    prediction_col = model_name.replace(" ", "_").replace("Regressor", "").strip("_")
    predictions_df[f"y_pred_{prediction_col}"] = y_pred_test

metrics_df = pd.DataFrame(metrics_rows)

print("Ensemble Model Performance")
print(metrics_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

## 7. Visualize Results

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test, label="Actual", linewidth=2)
plt.plot(predictions_df["y_pred_Random_Forest"], label="Random Forest Predicted", linestyle="--", linewidth=1.8)
plt.plot(predictions_df["y_pred_XGBoost"], label="XGBoost Predicted", linestyle=":", linewidth=2)
plt.xlabel("Test Sample")
plt.ylabel("Scaled SST")
plt.title("Ensemble Predictions vs Actual Test Values")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(data_dir / "ensemble_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
metrics_plot = metrics_df.set_index("Model")[["RMSE_Test", "MAE_Test", "R2_Test"]]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, metrics_plot.columns):
    metrics_plot[metric].plot(kind="bar", ax=ax, color=["steelblue", "tomato"])
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(data_dir / "ensemble_metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Test Predictions and Metrics

In [ ]:
predictions_df.to_csv(data_dir / "ensemble_test_predictions.csv", index=False)
metrics_df.to_csv(data_dir / "ensemble_metrics.csv", index=False)

display(metrics_df)
print("Saved predictions to:", (data_dir / "ensemble_test_predictions.csv").resolve())
print("Saved metrics to:", (data_dir / "ensemble_metrics.csv").resolve())